<a href="https://colab.research.google.com/github/Ayeshameer12/AI-Based-kidney-detector-application/blob/main/notebooks/train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kidney Ultrasound/CT Classifier — Normal vs Abnormal

Run this notebook in **Google Colab** with a GPU runtime (Runtime → Change runtime type → GPU).

**Before running:** zip your organized dataset (from `organize_dataset.py`) and upload it to your Google Drive, e.g. as `MyDrive/kidney_dataset.zip`, with this structure inside:
```
kidney_dataset/
    Normal/      *.jpg
    Abnormal/    *.jpg
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/kidney_dataset.zip'  # change if needed
EXTRACT_PATH = '/content/data'

os.makedirs(EXTRACT_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_PATH)

print(os.listdir(EXTRACT_PATH))

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# IMPORTANT: point this to the folder that directly contains Normal/ and Abnormal/
DATA_DIR = '/content/data/kidney_dataset'  # adjust based on your zip's internal folder name
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

## 2. Load data and split (train/val/test = 70/15/15)

In [ ]:
full_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='binary',
    class_names=['Normal', 'Abnormal'],
    image_size=IMG_SIZE,
    batch_size=None,
    shuffle=True,
    seed=SEED
)

n = tf.data.experimental.cardinality(full_ds).numpy()
train_n = int(0.7 * n)
val_n = int(0.15 * n)

train_ds = full_ds.take(train_n)
remaining = full_ds.skip(train_n)
val_ds = remaining.take(val_n)
test_ds = remaining.skip(val_n)

print(f'Train: {train_n}, Val: {val_n}, Test: {n - train_n - val_n}')

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Data augmentation + preprocessing

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

## 4. Handle class imbalance
Your Abnormal class has more images than Normal — compute class weights so the model doesn't get biased toward Abnormal.

In [ ]:
import pathlib
data_path = pathlib.Path(DATA_DIR)
n_normal = len(list((data_path / 'Normal').glob('*.jpg')))
n_abnormal = len(list((data_path / 'Abnormal').glob('*.jpg')))
total = n_normal + n_abnormal

class_weight = {
    0: total / (2 * n_normal),     # Normal
    1: total / (2 * n_abnormal),   # Abnormal
}
print('Normal:', n_normal, 'Abnormal:', n_abnormal)
print('Class weights:', class_weight)

## 5. Build the model (transfer learning — MobileNetV2)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze for initial training

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

## 6. Train (phase 1: frozen base)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks
)

## 7. Fine-tune (phase 2: unfreeze top layers of base model)

In [ ]:
base_model.trainable = True
# Freeze all but the last ~30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks
)

## 8. Plot training curves

In [ ]:
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(acc, label='train'); axes[0].plot(val_acc, label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(loss, label='train'); axes[1].plot(val_loss, label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.show()

## 9. Evaluate on the held-out test set

In [ ]:
y_true = []
y_pred_probs = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:\n', cm)

fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend()
plt.show()

## 10. Save the model

In [ ]:
model.save('/content/drive/MyDrive/kidney_model.h5')
print('Model saved to Google Drive as kidney_model.h5')
print('Download this file and place it in your Flask app under app/model/kidney_model.h5')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf

H5 = "/content/drive/MyDrive/kidney_model.h5"
if not os.path.exists(H5):
    H5 = "/content/kidney_model.h5"
print("using:", H5, "| exists:", os.path.exists(H5))

inp = tf.keras.Input((224, 224, 3))
x = inp / 127.5 - 1.0
base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights=None)
x = base(x)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
x = tf.keras.layers.Dropout(0.3, name="dropout")(x)
out = tf.keras.layers.Dense(1, activation="sigmoid", name="dense")(x)
m = tf.keras.Model(inp, out)
print("base name:", base.name)

m.load_weights(H5, by_name=True)
m.save("/content/kidney_model.keras")
print("saved")

ValueError: mount failed

In [4]:
!ls -lh /content/kidney_model.h5rw$0

-rw-r--r-- 1 root root 21M Nov  5 16:48 /content/kidney_model.h5


In [8]:
import os
import h5py
import numpy as np
import tensorflow as tf

H5 = "/content/kidney_model.h5"
print("exists:", os.path.exists(H5))

# 1) Architecture dobara banao
inp = tf.keras.Input((224, 224, 3))
x = inp / 127.5 - 1.0
base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights=None)
x = base(x)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
x = tf.keras.layers.Dropout(0.3, name="dropout")(x)
out = tf.keras.layers.Dense(1, activation="sigmoid", name="dense")(x)
m = tf.keras.Model(inp, out)
print("base name:", base.name)

# 2) h5 se saare weights naam ke hisaab se nikalo
f = h5py.File(H5, "r")
mw = f["model_weights"] if "model_weights" in f else f
print("groups:", list(mw.keys()))

found = {}
def visit(name, obj):
    if isinstance(obj, h5py.Dataset):
        parts = name.split("/")
        found[(parts[-2], parts[-1].split(":")[0])] = np.array(obj)

mw["mobilenetv2_1.00_224"].visititems(visit)
print("arrays mile:", len(found))

# 3) Har layer ke weights naam se set karo
missing = []
done = 0
for layer in base.layers:
    if not layer.weights:
        continue
    new = []
    for w in layer.weights:
        vn = w.name
        arr = found.get((layer.name, vn))
        if arr is None and vn == "kernel":
            arr = found.get((layer.name, "depthwise_kernel"))
        if arr is None and vn == "depthwise_kernel":
            arr = found.get((layer.name, "kernel"))
        new.append(arr)
    if any(a is None for a in new):
        missing.append(layer.name)
    else:
        layer.set_weights(new)
        done += 1
print("layers set:", done, "| missing:", missing[:10])

# 4) Aakhri Dense layer ke weights
dv = {}
def v2(name, obj):
    if isinstance(obj, h5py.Dataset):
        dv[name.split("/")[-1].split(":")[0]] = np.array(obj)
mw["dense"].visititems(v2)
m.get_layer("dense").set_weights([dv["kernel"], dv["bias"]])
print("dense set, kernel shape:", dv["kernel"].shape)

# 5) Naye format mein save
m.save("/content/kidney_model.keras")
print("saved")

exists: True
base name: mobilenetv2_1.00_224
groups: ['dense', 'dropout', 'global_average_pooling2d', 'input_layer_1', 'mobilenetv2_1.00_224', 'sequential', 'top_level_model_weights']
arrays mile: 260
layers set: 104 | missing: []
dense set, kernel shape: (1280, 1)
saved


In [9]:
!ls -lh /content

total 30M
-rw-r--r-- 1 root root  21M Nov  5 16:48 kidney_model.h5
-rw-r--r-- 1 root root 9.2M Nov  5 16:55 kidney_model.keras
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [13]:
%%writefile app.py
import os

import gradio as gr
import numpy as np
import tensorflow as tf
from matplotlib import colormaps
from PIL import Image

MODEL_PATH = os.environ.get("MODEL_PATH", "kidney_model.keras")
IMG_SIZE = (224, 224)

# Index 0 = Normal, index 1 = Abnormal -> sigmoid output = P(Abnormal)
BASE_NAME = "mobilenetv2_1.00_224"
DISCLAIMER = (
    "⚠️ This is a student/research screening tool, **not a medical diagnosis**. "
    "Always consult a doctor / radiologist."
)

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
base = model.get_layer(BASE_NAME)
gap = model.get_layer("global_average_pooling2d")
drop = model.get_layer("dropout")
head = model.get_layer("dense")


def to_array(pil_img):
    img = pil_img.convert("RGB").resize(IMG_SIZE)
    return np.asarray(img, dtype=np.float32)


def predict_prob(arr):
    return float(model(arr[None], training=False).numpy().ravel()[0])


def grad_cam(arr):
    x = tf.convert_to_tensor(arr[None] / 127.5 - 1.0)
    with tf.GradientTape() as tape:
        conv_out = base(x, training=False)
        tape.watch(conv_out)
        score = head(drop(gap(conv_out), training=False), training=False)[:, 0]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * conv_out, axis=-1))[0].numpy()
    cam = cam / (cam.max() + 1e-8)
    return cam, float(score.numpy()[0])


def overlay_heatmap(pil_img, cam, alpha=0.4):
    img = pil_img.convert("RGB").resize(IMG_SIZE)
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize(IMG_SIZE, Image.BILINEAR)
    heat = colormaps["jet"](np.asarray(cam_img) / 255.0)[..., :3]
    base_img = np.asarray(img, dtype=np.float32) / 255.0
    out = (1 - alpha) * base_img + alpha * heat
    return Image.fromarray(np.uint8(np.clip(out, 0, 1) * 255))


_test = np.random.randint(0, 255, (*IMG_SIZE, 3)).astype(np.float32)
_, _manual = grad_cam(_test)
_full = predict_prob(_test)
if abs(_manual - _full) > 1e-3:
    print(f"[WARN] Grad-CAM path ({_manual:.4f}) != model output ({_full:.4f}).")
else:
    print("[OK] Grad-CAM path matches the model output.")


def analyze(pil_img, threshold):
    if pil_img is None:
        return None, None, "Please upload an image first."

    arr = to_array(pil_img)
    p_abnormal = predict_prob(arr)
    p_normal = 1.0 - p_abnormal
    cam, _ = grad_cam(arr)
    heat = overlay_heatmap(pil_img, cam)

    is_abnormal = p_abnormal >= threshold
    verdict = "Abnormal" if is_abnormal else "Normal"
    conf = p_abnormal if is_abnormal else p_normal
    summary = (
        f"### Result: **{verdict}**\n"
        f"Confidence: **{conf * 100:.1f}%**  \n"
        f"P(Abnormal) = {p_abnormal:.3f} | threshold = {threshold:.2f}\n\n"
        "Red/yellow areas in the heatmap are the regions that pushed the model "
        "most towards “Abnormal”.\n\n" + DISCLAIMER
    )
    return {"Abnormal": p_abnormal, "Normal": p_normal}, heat, summary


with gr.Blocks(title="Kidney Normal vs Abnormal Classifier") as demo:
    gr.Markdown("# 🩻 AI Based Kidney Detector: Normal vs Abnormal\nUpload an ultrasound/CT image to get a prediction and a Grad-CAM heatmap.")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="Upload scan image")
            thr = gr.Slider(
                0.1, 0.9, value=0.5, step=0.05,
                label="Abnormal threshold (lower = fewer missed abnormal cases, more false alarms)",
            )
            btn = gr.Button("Analyze", variant="primary")
        with gr.Column():
            lab = gr.Label(label="Probabilities")
            cam_out = gr.Image(label="Grad-CAM heatmap")
    msg = gr.Markdown()
    btn.click(analyze, [inp, thr], [lab, cam_out, msg])

if __name__ == "__main__":
    demo.launch(share=True)

Overwriting app.py


In [14]:
!ls -lh /content

total 30M
-rw-r--r-- 1 root root 3.8K Nov  5 17:00 app.py
-rw-r--r-- 1 root root  21M Nov  5 16:48 kidney_model.h5
-rw-r--r-- 1 root root 9.2M Nov  5 16:55 kidney_model.keras
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [ ]:
!pip install -q gradio
!python app.py

I0000 00:00:1793898088.086658    7863 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793898091.641267    7863 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1793898093.800795    7863 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
[OK] Grad-CAM path matches the model output.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://fee1e547db448ae1bd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
